# Building an End-to-End RAG Evaluation

This notebook is a hands-on, step-by-step lab to build and evaluate a complete Retrieval-Augmented Generation (RAG) pipeline.

You will learn how to:

1. Build an end-to-end RAG pipeline from a PDF.
2. Create a ground truth dataset for evaluation.
3. Measure **Faithfulness** and **Answer Relevance** to reduce hallucinations.
4. Interpret results and iterate on the pipeline

> Recommended workflow: run one cell at a time, read the explanation, and inspect the outputs before moving on.

## Step 0 - Install dependencies

This workshop uses:

- **LangChain** for pipeline building
- **Chroma** as the vector database
- **Ragas** for RAG evaluation metrics
- **Ollama** for RAG answers: the notebook calls `ollama_model_runner.py` with `subprocess` (same pattern as `Module_4_Building_a_comprehensive_RAG_system.ipynb`)
- **Sentence Transformers** via `langchain-huggingface` / `HuggingFaceEmbeddings` for indexing and Ragas (default `sentence-transformers/all-MiniLM-L6-v2`, same as the Module 4 lab)

If you already installed these packages, you can skip the next cell.

In [2]:
# Run once per environment
%pip install -q -U langchain langchain-community langchain-openai langchain-huggingface sentence-transformers chromadb pypdf pandas numpy tqdm ragas datasets

Note: you may need to restart the kernel to use updated packages.


## Step 1 - Imports and environment setup

We keep this workshop explicit and transparent:

- One small code block per action
- Printed outputs for quick sanity checks
- Reusable helper functions

In [3]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

pd.set_option("display.max_colwidth", 150)

/home/ethan/anaconda3/envs/kmouc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configure model endpoints

**RAG generation (answers)** uses **Ollama** by running `ollama_model_runner.py` from this folder (or `Module_4/ollama_model_runner.py` if your working directory is the repo root). Start `ollama serve` and `ollama pull <model>` outside the notebook.

**Example** python Module_4/ollama_model_runner.py --host http://localhost:11434 --models llama3.1:8b --prompt-file /tmp/tmpivn3qis0.txt --temperature 0.2 --max-tokens 320

**Embeddings** (Chroma index and Ragas) use **Sentence Transformers** through LangChain's `HuggingFaceEmbeddings`, aligned with `Module_4_Building_a_comprehensive_RAG_system.ipynb` (default model name: `sentence-transformers/all-MiniLM-L6-v2`). Override with env var `RAG_EMBED_MODEL` if you want another HF model id.

> Edit the next two code cells if your Ollama host, chat model, or embedding model id differ.

In [4]:
# --- Ollama: used by ollama_model_runner.py for RAG answers (subprocess) ---
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODELS = os.getenv("OLLAMA_MODELS", "llama3.1:8b")  # comma-separated if you ever pass several
OLLAMA_TEMPERATURE = float(os.getenv("OLLAMA_TEMPERATURE", "0.2"))
OLLAMA_MAX_TOKENS = int(os.getenv("OLLAMA_MAX_TOKENS", "512"))

# --- Sentence Transformers: same default as Module_4_Building_a_comprehensive_RAG_system.ipynb ---
EMBED_MODEL_NAME = os.getenv("RAG_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")


def _resolve_ollama_runner_path() -> Path:
    for candidate in (Path("ollama_model_runner.py"), Path("Module_4") / "ollama_model_runner.py"):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "ollama_model_runner.py not found next to the notebook or under Module_4/. "
        "Open this notebook from Module_4 or clone the file into your cwd."
    )


OLLAMA_RUNNER = _resolve_ollama_runner_path()
OLLAMA_CHAT_MODEL = OLLAMA_MODELS.split(",")[0].strip()

print("OLLAMA_HOST:", OLLAMA_HOST)
print("OLLAMA_RUNNER:", OLLAMA_RUNNER)
print("OLLAMA_MODELS:", OLLAMA_MODELS)
print("OLLAMA_CHAT_MODEL (Ragas -> Ollama /v1):", OLLAMA_CHAT_MODEL)
print("EMBED_MODEL_NAME (Sentence Transformers):", EMBED_MODEL_NAME)

OLLAMA_HOST: http://localhost:11434
OLLAMA_RUNNER: /home/ethan/newgen/KMOU_Course/Module_4/ollama_model_runner.py
OLLAMA_MODELS: llama3.1:8b
OLLAMA_CHAT_MODEL (Ragas -> Ollama /v1): llama3.1:8b
EMBED_MODEL_NAME (Sentence Transformers): sentence-transformers/all-MiniLM-L6-v2


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    encode_kwargs={"normalize_embeddings": True},
)

# Ragas calls an LLM over HTTP; use Ollama's OpenAI-compatible endpoint (same chat model as the runner).
ragas_llm = ChatOpenAI(
    model=OLLAMA_CHAT_MODEL,
    api_key=os.getenv("OPENAI_API_KEY", "ollama"),
    base_url=f"{OLLAMA_HOST.rstrip('/')}/v1",
    temperature=0,
)


def call_ollama_runner(prompt: str, *, models: str | None = None) -> str:
    """Run ollama_model_runner.py (same argv style as Module_4_Building_a_comprehensive_RAG_system.ipynb).

    Pass ``models=`` to override ``OLLAMA_MODELS`` (for example a dedicated labeler model in Step 6b).
    """
    models_arg = models if models is not None else OLLAMA_MODELS
    pf = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8")
    path = pf.name
    try:
        pf.write(prompt)
        pf.close()
        cmd = [
            sys.executable,
            str(OLLAMA_RUNNER),
            "--host",
            OLLAMA_HOST,
            "--models",
            models_arg,
            "--prompt-file",
            path,
            "--temperature",
            str(OLLAMA_TEMPERATURE),
            "--max-tokens",
            str(OLLAMA_MAX_TOKENS),
        ]
        run = subprocess.run(cmd, capture_output=True, text=True)
        if run.returncode != 0:
            raise RuntimeError(
                f"ollama_model_runner.py exit {run.returncode}\nSTDERR:\n{run.stderr[:4000]}"
            )
        try:
            payload = json.loads(run.stdout)
        except json.JSONDecodeError as e:
            raise RuntimeError(
                f"Invalid JSON from ollama_model_runner.py: {e}\nSTDOUT:\n{run.stdout[:2000]}"
            ) from e
        outputs = payload.get("outputs") or []
        if not outputs:
            raise RuntimeError("Runner JSON contained no outputs")
        first = outputs[0]
        if first.get("status") != "ok":
            raise RuntimeError(f"Ollama runner error: {first.get('status')}")
        return str(first.get("response", "")).strip()
    finally:
        try:
            os.unlink(path)
        except OSError:
            pass


print("Embeddings (HuggingFace), ragas_llm, and call_ollama_runner() are ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5757.34it/s]


Embeddings (HuggingFace), ragas_llm, and call_ollama_runner() are ready.


## Step 2 - Load and inspect the PDF document

We now load: `data/IGF Code (124Pages).pdf`.

At this step, we only validate:

- File exists
- Number of pages loaded
- Sample text quality

In [6]:
pdf_path = Path("data") / "IGF Code (124Pages).pdf"
print("PDF path:", pdf_path)
print("Exists:", pdf_path.exists())

PDF path: data/IGF Code (124Pages).pdf
Exists: True


In [7]:
loader = PyPDFLoader(str(pdf_path))
docs = loader.load()
print("Loaded pages:", len(docs))

Loaded pages: 124


In [8]:
sample_page = 0
print(docs[sample_page].page_content[:1500])

MSC 95/22/Add.1 
Annex 1, page 1 
 
 
I:\MSC\95\MSC 95-22-ADD.1 (E).docx 
ANNEX 1 
 
RESOLUTION MSC.391(95)  
(adopted on 11 June 2015) 
 
ADOPTION OF THE INTERNATIONAL CODE OF SAFETY FOR SHIPS USING GASES 
OR OTHER LOW-FLASHPOINT FUELS (IGF CODE) 
 
 
THE MARITIME SAFETY COMMITTEE, 
 
RECALLING Article 28(b) of the Convention on the International Maritime Organization 
concerning the function of the Committee, 
 
RECOGNIZING the need for a mandatory code for ships using gases or other low -flashpoint 
fuels, 
 
NOTING resolution MSC .392(95), by which it adopted, inter alia,  amendments to 
chapters II-1,II-2 and the appendix to the annex of the International Convention for the Safety 
of Life at Sea, 1974  ("the Convention"), to make the provisions of the International Code of 
Safety for Ships using Gases or other  Low-flashpoint Fuels (IGF Code) mandatory under the 
Convention, 
 
HAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for 
Ships usin

## Step 3 - Chunking strategy for retrieval

RAG quality depends heavily on chunking.

We start with a practical baseline:

- `chunk_size = 1000`
- `chunk_overlap = 150`

Later, you can tune this and compare evaluation scores.

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print("Total chunks:", len(chunks))

Total chunks: 400


In [10]:
ROW_NUMBER = 20
chunk_preview = pd.DataFrame(
    {
        "chunk_id": list(range(min(ROW_NUMBER, len(chunks)))),
        "page": [chunks[i].metadata.get("page") for i in range(min(ROW_NUMBER, len(chunks)))],
        "text_preview": [chunks[i].page_content[:220].replace("\n", " ") for i in range(min(ROW_NUMBER, len(chunks)))],
    }
)
chunk_preview

,chunk_id,page,text_preview
0,0,0,"MSC 95/22/Add.1 Annex 1, page 1 I:\MSC\95\MSC 95-22-ADD.1 (E).docx ANNEX 1 RESOLUTION MSC.391(95) (adopted on 11 June 2015) ADOPTIO..."
1,1,0,"Convention, HAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for Ships using Gases or other Low-flashpoi..."
2,2,0,"5 REQUESTS the Secretary-General of the Organization to transmit certified copies of the present resolution and the text of the IGF Code, contain..."
3,3,1,"MSC 95/22/Add.1 Annex 1, page 2 I:\MSC\95\MSC 95-22-ADD.1 (E).docx ANNEX INTERNATIONAL CODE OF SAFETY FOR SHIPS USING GASES OR OTHER LO..."
4,4,1,2.3 Alternative design ................................................................................................ 11 3 GOAL AND FUNCTIONAL ...
5,5,1,4.3 Limitation of explosion consequences .............................................................. 13 PART A-1 SPECIFIC REQUIREMENTS FOR SHI...
6,6,1,5.6 Regulations for ESD-protected machinery spaces ........................................... 18 5.7 Regulations for location and protection of ...
7,7,1,6 FUEL CONTAINMENT SYSTEM ............................................................................ 21 6.1 Goal .................................
8,8,2,"MSC 95/22/Add.1 Annex 1, page 3 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 6.2 Functional requirements ..............................................."
9,9,2,6.9 Regulations for the maintaining of fuel storage condition .............................. 62 6.10 Regulations on atmospheric control within th...


## Step 4 - Build or load vector index (Chroma)

Chunks are embedded and stored in a **local** Chroma database under `outputs/chroma_igf_rag_eval/`.

- **First run (no folder / empty):** builds the index from `chunks`, then saves to disk.
- **Later runs:** loads the existing index from disk so you skip re-embedding (much faster).
- **Rebuild from scratch:** set `FORCE_REBUILD_CHROMA = True` in the next code cell, or export env `FORCE_REBUILD_CHROMA=1` (needed if you change chunking, the PDF, or the embedding model so vectors stay consistent).

On the **load** path you do not need `chunks` in memory; you still need the same `embeddings` object (same model) as when the store was built. For the **build** path, run the earlier cells that define `chunks` and `embeddings`.

In [11]:
import shutil

CHROMA_COLLECTION = "igf_code_rag_eval"
chroma_dir = Path("outputs") / "chroma_igf_rag_eval"

# Set True in-notebook to wipe disk and rebuild from `chunks` (or export FORCE_REBUILD_CHROMA=1).
FORCE_REBUILD_CHROMA = False
FORCE_REBUILD_CHROMA = FORCE_REBUILD_CHROMA or os.getenv("FORCE_REBUILD_CHROMA", "0").lower() in (
    "1",
    "true",
    "yes",
)


def _chroma_persist_ready(path: Path) -> bool:
    """True if a Chroma persistent store already exists on disk."""
    if not path.is_dir():
        return False
    return (path / "chroma.sqlite3").is_file()


chroma_dir.mkdir(parents=True, exist_ok=True)

if FORCE_REBUILD_CHROMA and chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    chroma_dir.mkdir(parents=True, exist_ok=True)

if _chroma_persist_ready(chroma_dir) and not FORCE_REBUILD_CHROMA:
    vector_db = Chroma(
        collection_name=CHROMA_COLLECTION,
        embedding_function=embeddings,
        persist_directory=str(chroma_dir),
    )
    n = vector_db._collection.count()
    print(f"Loaded existing Chroma DB from disk ({n} documents).")
else:
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=CHROMA_COLLECTION,
        persist_directory=str(chroma_dir),
    )
    print(f"Built Chroma DB from chunks and saved to disk ({len(chunks)} documents).")

print("Chroma directory:", chroma_dir.resolve())

/tmp/ipykernel_20986/3521674612.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


Loaded existing Chroma DB from disk (400 documents).
Chroma directory: /home/ethan/newgen/KMOU_Course/Module_4/outputs/chroma_igf_rag_eval


In [12]:
retriever = vector_db.as_retriever(search_kwargs={"k": 4})
print("Retriever top-k:", retriever.search_kwargs.get("k"))

Retriever top-k: 4


## Step 5 - Build a simple RAG answer function

We define a minimal but robust RAG function:

1. Retrieve top-k chunks
2. Build a context block
3. Ask the model to answer **only from the context**
4. Return answer + retrieved contexts (needed for evaluation)

In [13]:
RAG_PROMPT_TEMPLATE = """
You are a strict assistant.
Answer the question using ONLY the provided context.
If the context does not contain enough information, say: "I don't have enough information from the provided document."

Question:
{question}

Context:
{context}
""".strip()

In [14]:
def answer_with_rag(question: str, retriever):
    retrieved_docs = retriever.invoke(question)
    contexts = [d.page_content for d in retrieved_docs]
    joined_context = "\n\n---\n\n".join(contexts)

    prompt = RAG_PROMPT_TEMPLATE.format(question=question, context=joined_context)
    answer_text = call_ollama_runner(prompt)

    return {
        "question": question,
        "answer": answer_text,
        "contexts": contexts,
    }

In [15]:
test_question = "What is the purpose of the IGF Code and who is expected to follow it?"
example_result = answer_with_rag(test_question, retriever)

print("Question:", example_result["question"])
print("\nAnswer:\n", example_result["answer"])
print("\nRetrieved contexts:", len(example_result["contexts"]))

Question: What is the purpose of the IGF Code and who is expected to follow it?

Answer:
 The purpose of the IGF Code is to address all areas that need special consideration for the usage of low-flashpoint fuel. The basic philosophy of the IGF Code considers a goal-based approach, specifying goals and functional requirements for each section forming the basis for design, construction, and operation.

Contracting Governments to the Convention are expected to follow the IGF Code.

Retrieved contexts: 4


## Step 6 - Ground truth dataset (overview)

A **ground truth** row pairs a **`question`** with a **`ground_truth`** string that stands in for the **correct reference answer** when you score the RAG system. If labels are weak or biased, metrics tell you more about the label set than about your pipeline.

**Scale (production):** required volume depends on domain, risk, and budget; teams may label **tens to millions** of items. **Coverage** (topics, difficulty, edge cases) matters as much as raw count, provided labeling is consistent and audited.

**Per-row fields (used later in Steps 7–8):**

- `question` — what you ask the RAG system under test  
- `ground_truth` — the reference answer Ragas compares to the model answer (**manual** labels in Step 6a, plus an optional **LLM-synthetic** table from Step 6b scored separately from Step 7 onward)

This step is split so you can choose **how** labels are produced:

- **Step 6a — Method 1 (manual):** human-written labels exported to **`ground_truth_dataset_manual.csv`**; **Step 7 loads that file** (plus the LLM draft CSV) instead of using in-memory Python lists.  
- **Step 6b — Method 2 (LLM synthetic from the PDF):** you only set **how many** rows to generate. The LLM first completes **all questions** from random **PDF chunks** (`chunks` from Step 3), **then** (separate pass) writes **answers** from those frozen questions plus the **same** chunk text. **Step 6a is not required** for Method 2—only Step 3 so `chunks` exists.

#### Method 1 vs Method 2 — strengths and weaknesses

| | **Method 1 — Manual (6a)** | **Method 2 — LLM on PDF chunks (6b)** |
|---|---------------------------|----------------------------------------|
| **Strengths** | Highest fidelity and auditability; avoids “the model grading itself”; best for regulated or high-risk domains. | Fast to bootstrap many (question, answer) pairs; questions are grounded in **real** chunk text you sampled; good for drills and coverage exploration. |
| **Weaknesses** | Slow and expensive at scale; needs guidelines and (ideally) a second reviewer for consistency. | Labels are **drafts** until a human checks them against the PDF; risk of hallucination or soft paraphrase; if the labeler model equals the RAG model you add **circular bias** (mitigate with `OLLAMA_GT_LABEL_MODEL` or a separate teacher). |
| **Depends on** | You reading the source PDF (or rubric). | Step 3 chunking (`chunks`); local Ollama in 6b (or optional OpenAI path in 6c). |

**Optional Step 6c** repeats the **same two-phase recipe** as 6b using a hosted `ChatOpenAI` labeler when API credentials are enabled—still independent of Step 6a.


### Step 6a — Method 1: Manual ground-truth curation

**Goal:** build a **trusted** evaluation table by reading the source material (here: the IGF Code PDF) and writing questions plus short factual answers yourself (or from an instructor-provided rubric).

| Topic | Details |
|-------|---------|
| **Workflow** | Read passage → write `question` → write `ground_truth` strictly from that passage → (recommended) add private notes with page/section → optional second reviewer. |
| **Strengths** | Highest label fidelity; avoids "the model grading itself"; mandatory for high-risk domains. |
| **Limitations** | Slow and costly at scale; needs annotation guidelines so multiple people label consistently. |

The list in the next cell is a **starter template** only. Replace it with evidence-backed items from the PDF when you run a serious evaluation.

In [16]:
from IPython.display import display

# --- Step 6a: manually curated ground truth (saved to CSV; Step 7 loads that file) ---
manual_ground_truth_data = [
    {
        "question": "What is the main objective of the IGF Code?",
        "ground_truth": "The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework.",
        "label_source": "manual",
    },
    {
        "question": "Who are the intended stakeholders or users of the IGF Code?",
        "ground_truth": "The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",
        "label_source": "manual",
    },
    {
        "question": "What kinds of behavior or practices does the Code try to prevent?",
        "ground_truth": "It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",
        "label_source": "manual",
    },
    {
        "question": "Does the Code include governance or compliance expectations? If yes, what is the intent?",
        "ground_truth": "Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",
        "label_source": "manual",
    },
    {
        "question": "How does the Code describe accountability when rules are not followed?",
        "ground_truth": "The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",
        "label_source": "manual",
    },
    {
        "question": "What is the expected benefit of following the IGF Code consistently?",
        "ground_truth": "Consistent adherence improves trust, transparency, and responsible decision-making across the organizations and participants covered by the Code.",
        "label_source": "manual",
    },
]

ground_truth_data = manual_ground_truth_data
ground_truth_df = pd.DataFrame(ground_truth_data)

# Step 7 reads ``outputs/ground_truth_dataset_manual.csv`` and ``outputs/ground_truth_dataset_llm_draft.csv`` (run the export cell above first).
# You can still use ``ground_truth_data`` / ``llm_draft_ground_truth_df`` in-session; CSV is the source of truth for evaluation reruns.

print("Manual ground-truth rows:", len(ground_truth_df))
display(ground_truth_df)


Manual ground-truth rows: 6


,question,ground_truth,label_source
0,What is the main objective of the IGF Code?,"The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework.",manual
1,Who are the intended stakeholders or users of the IGF Code?,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",manual
2,What kinds of behavior or practices does the Code try to prevent?,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",manual
3,"Does the Code include governance or compliance expectations? If yes, what is the intent?","Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",manual
4,How does the Code describe accountability when rules are not followed?,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",manual
5,What is the expected benefit of following the IGF Code consistently?,"Consistent adherence improves trust, transparency, and responsible decision-making across the organizations and participants covered by the Code.",manual


### Step 6b — Method 2: LLM synthetic ground truth from the PDF (local Ollama)

**Goal:** generate **new** evaluation `(question, ground_truth)` rows from **actual PDF text** already split into `chunks` (Step 3). You set **`N_SYNTH_GROUND_TRUTH`** (count only); the notebook samples that many chunks at random.

**Important — two explicit LLM phases (questions first, then answers):**

1. **Phase 1 — Questions only**  
   For each sampled chunk, the labeler LLM writes **one question** answerable from that chunk alone. **No answer prompts run in this phase**—the run “stops” after all questions exist.

2. **Phase 2 — Answers from frozen questions**  
   Using the **same** chunk text paired with the **Phase 1 question** (no re-sampling), the labeler LLM writes `ground_truth`. You can inspect the question table between phases in the code output.

This design matches a common evaluation workflow: **freeze the question set**, then generate or revise reference answers, so answers are always conditioned on a fixed question + passage pair.

#### Labeler LLM vs RAG generator LLM (bias)

Prefer a **different** model or provider for labeling than for RAG answers (`OLLAMA_GT_LABEL_MODEL`). Reusing the same weights is a **lesson shortcut**, not ideal for publication-grade benchmarks.

#### Environment knobs

| Variable | Meaning |
|----------|---------|
| `N_SYNTH_GROUND_TRUTH` | How many synthetic rows (each = 1 chunk, **2** LLM calls: Q then A). |
| `SYNTH_GROUND_TRUTH_SEED` | RNG seed for which chunks are sampled. |
| `OLLAMA_GT_LABEL_MODEL` | Optional dedicated Ollama tag for both phases. |
| `SKIP_STEP6B_SYNTH` or `SKIP_LLM_DRAFT_GT` | Set `1` to skip all Ollama calls in Step 6b. |


In [17]:
# --- Step 6b: synthetic (question, ground_truth) from PDF chunks — two phases: ALL questions first, THEN answers ---
import inspect
import random

# === How many synthetic rows (each row = 1 random chunk, phase1 question + phase2 answer) ===
N_SYNTH_GROUND_TRUTH = 20
N_SYNTH_GROUND_TRUTH = int(
    os.getenv(
        "N_SYNTH_GROUND_TRUTH",
        os.getenv("N_STEP6B_SYNTH_PAIRS", str(N_SYNTH_GROUND_TRUTH)),
    )
)
SYNTH_GROUND_TRUTH_SEED = int(os.getenv("SYNTH_GROUND_TRUTH_SEED", "42"))
OLLAMA_GT_LABEL_MODEL = os.getenv("OLLAMA_GT_LABEL_MODEL", "").strip()

SYNTH_Q_PROMPT = """
You write evaluation questions for a regulatory / technical document.
Given PASSAGE text only, write ONE clear, specific question that can be answered using ONLY information in the passage.
Rules:
- Output ONLY the question text (no numbering, no quotes, no preamble).
- Do not mention "the passage" or "the document".

PASSAGE:
{passage}
""".strip()


SYNTH_A_PROMPT = """
You write reference answers for RAG evaluation (ground-truth style labels).
Given PASSAGE and a QUESTION, write ONE concise factual paragraph answering the question using ONLY the passage.
Rules:
- If the passage does not support an answer, reply with exactly this single line: INSUFFICIENT_CONTEXT
- Max ~120 words, plain text, no bullet list.

QUESTION:
{question}

PASSAGE:
{passage}
""".strip()



def _trim_passage(passage: str, limit: int = 12000) -> str:
    t = passage.strip()
    if len(t) > limit:
        return t[:limit] + "\n\n[...truncated for labeling prompt...]"
    return t


def _call_ollama_runner_for_gt_labeling(prompt: str, labeler_models: str | None) -> str:
    sig = inspect.signature(call_ollama_runner)
    if "models" in sig.parameters:
        return call_ollama_runner(prompt, models=labeler_models)
    if labeler_models:
        raise RuntimeError(
            "OLLAMA_GT_LABEL_MODEL is set but call_ollama_runner has no models= argument. "
            "Re-run the setup cell that defines call_ollama_runner, or unset OLLAMA_GT_LABEL_MODEL."
        )
    return call_ollama_runner(prompt)


SKIP_STEP6B_SYNTH = os.getenv("SKIP_STEP6B_SYNTH", os.getenv("SKIP_LLM_DRAFT_GT", "0")).lower() in (
    "1",
    "true",
    "yes",
)

_labeler_models = OLLAMA_GT_LABEL_MODEL or None
if _labeler_models is None:
    print("OLLAMA_GT_LABEL_MODEL is empty — synthesizer uses OLLAMA_MODELS (lesson shortcut).")
else:
    print("Using dedicated synthesizer model:", _labeler_models)

if "chunks" not in globals() or not chunks:
    raise RuntimeError("Run Step 3 (chunking) first so `chunks` exists — chunks come from the PDF.")

if SKIP_STEP6B_SYNTH:
    llm_draft_ground_truth_data = []
    llm_draft_ground_truth_df = pd.DataFrame(
        columns=[
            "question",
            "ground_truth",
            "source_page",
            "passage_chars",
            "label_source",
            "labeler_models",
        ]
    )
    print("SKIP_STEP6B_SYNTH / SKIP_LLM_DRAFT_GT is set — skipped Step 6b.")
else:
    rng = random.Random(SYNTH_GROUND_TRUTH_SEED)
    _chunk_list = list(chunks)
    n_pick = min(N_SYNTH_GROUND_TRUTH, len(_chunk_list))
    picked = rng.sample(_chunk_list, n_pick)
    passages_raw = [c.page_content for c in picked]
    page_hints = [c.metadata.get("page") for c in picked]
    passages = [_trim_passage(p) for p in passages_raw]

    # ----- Phase 1: generate ALL questions from real chunk text; no answer LLM calls yet -----
    synth_questions: list[str] = []
    for passage_trim, page_hint in tqdm(
        list(zip(passages, page_hints)),
        desc="Step 6b phase 1 — questions only (PDF chunks)",
    ):
        q_prompt = SYNTH_Q_PROMPT.format(passage=passage_trim)
        question = _call_ollama_runner_for_gt_labeling(q_prompt, _labeler_models).strip()
        synth_questions.append(question)

    print("Phase 1 complete:", len(synth_questions), "questions from", n_pick, "PDF chunks (LLM paused before answers).")
    display(
        pd.DataFrame(
            {
                "question": synth_questions,
                "source_page": page_hints,
                "passage_chars": [len(p) for p in passages],
            }
        )
    )

    # ----- Phase 2: answers only, each conditioned on frozen question + same passage -----
    rows: list[dict] = []
    for question, passage_trim, page_hint in tqdm(
        list(zip(synth_questions, passages, page_hints)),
        desc="Step 6b phase 2 — answers from frozen questions + passages",
    ):
        a_prompt = SYNTH_A_PROMPT.format(question=question, passage=passage_trim)
        ground_truth = _call_ollama_runner_for_gt_labeling(a_prompt, _labeler_models).strip()
        rows.append(
            {
                "question": question,
                "ground_truth": ground_truth,
                "source_page": page_hint,
                "passage_chars": len(passage_trim),
                "label_source": "step6b_synthetic_local",
                "labeler_models": _labeler_models or OLLAMA_MODELS,
            }
        )

    llm_draft_ground_truth_data = rows
    llm_draft_ground_truth_df = pd.DataFrame(rows)

print("Synthetic rows requested:", N_SYNTH_GROUND_TRUTH, "| produced:", len(llm_draft_ground_truth_df))
display(llm_draft_ground_truth_df)

if "openai_gt_draft_df" not in globals():
    openai_gt_draft_df = pd.DataFrame(
        columns=["question", "ground_truth", "source_page", "passage_chars", "label_source", "labeler_model"]
    )


OLLAMA_GT_LABEL_MODEL is empty — synthesizer uses OLLAMA_MODELS (lesson shortcut).


Step 6b phase 1 — questions only (PDF chunks): 100%|██████████| 20/20 [00:17<00:00,  1.11it/s]

Phase 1 complete: 20 questions from 20 PDF chunks (LLM paused before answers).


,question,source_page,passage_chars
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure?,101,965
1,"What is the minimum distance between fuel tanks in a cargo ship with a volume Vc greater than or equal to 30,000 m3?",16,946
2,What page number is referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces?,2,378
3,What is the minimum design condition under which a type expansion joint must withstand twice the design pressure?,116,961
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,43,994
5,What is the minimum design temperature at which a longitudinal bulkhead between liquefied gas fuel tanks must remain suitable for credit to be tak...,39,933
6,What are the six total stresses and shear stresses that shall be determined for calculating the equivalent stress σc?,35,990
7,"What potential dangers to the ship, persons on board, or environment should be avoided when designing the fuel containment system?",20,926
8,What percentage of butt-welded joints of pipes must be subjected to radiographic or ultrasonic inspection?,115,787
9,What is the minimum distance that the lowermost boundary of the fuel tank(s) must be located above?,14,504


Step 6b phase 2 — answers from frozen questions + passages: 100%|██████████| 20/20 [00:30<00:00,  1.55s/it]

Synthetic rows requested: 20 | produced: 20


,question,ground_truth,source_page,passage_chars,label_source,labeler_models
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure?,"The buckling strength analyses of fuel tanks subject to external pressure shall be carried out in accordance with recognized standards, which adeq...",101,965,step6b_synthetic_local,llama3.1:8b
1,"What is the minimum distance between fuel tanks in a cargo ship with a volume Vc greater than or equal to 30,000 m3?","For a cargo ship with a volume Vc greater than or equal to 30,000 m3, the minimum distance between fuel tanks is 2 meters, as specified in paragra...",16,946,step6b_synthetic_local,llama3.1:8b
2,What page number is referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces?,The page number referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces is 78.,2,378,step6b_synthetic_local,llama3.1:8b
3,What is the minimum design condition under which a type expansion joint must withstand twice the design pressure?,The minimum design condition under which a type expansion joint must withstand twice the design pressure is at the minimum design temperature. Thi...,116,961,step6b_synthetic_local,llama3.1:8b
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,43,994,step6b_synthetic_local,llama3.1:8b
5,What is the minimum design temperature at which a longitudinal bulkhead between liquefied gas fuel tanks must remain suitable for credit to be tak...,"For a longitudinal bulkhead between liquefied gas fuel tanks, credit can be taken for heating if the material remains suitable at a minimum design...",39,933,step6b_synthetic_local,llama3.1:8b
6,What are the six total stresses and shear stresses that shall be determined for calculating the equivalent stress σc?,The six total stresses and shear stresses that shall be determined for calculating the equivalent stress σc are: total normal stress in x-directio...,35,990,step6b_synthetic_local,llama3.1:8b
7,"What potential dangers to the ship, persons on board, or environment should be avoided when designing the fuel containment system?","When designing the fuel containment system, potential dangers to avoid include exposure of ship materials to temperatures below acceptable limits,...",20,926,step6b_synthetic_local,llama3.1:8b
8,What percentage of butt-welded joints of pipes must be subjected to radiographic or ultrasonic inspection?,At least 10% of butt-welded joints of pipes must be subjected to radiographic or ultrasonic inspection.,115,787,step6b_synthetic_local,llama3.1:8b
9,What is the minimum distance that the lowermost boundary of the fuel tank(s) must be located above?,"The minimum distance that the lowermost boundary of the fuel tank(s) must be located above is 2.0 m, or B/15, whichever is less.",14,504,step6b_synthetic_local,llama3.1:8b


### Optional — Step 6c: same two-phase synthetic ground truth (OpenAI API)

This block is **optional** and **off by default**. It mirrors **Step 6b**: **Phase 1** writes **all** questions from random PDF `chunks`, then **Phase 2** writes answers from those frozen questions plus the same chunk text—using `ChatOpenAI` instead of the local runner.

**No dependency on Step 6b:** prompt templates are defined again in the code cell below (duplicate of 6b text). You still need **Step 3** so `chunks` exists.

| Variable | Purpose |
|----------|---------|
| `USE_OPENAI_GT_LABELER` | Set `1` to run this block. |
| `OPENAI_API_KEY` | API key for the OpenAI-compatible service. |
| `N_SYNTH_GROUND_TRUTH` | Same row count as Step 6b (default can be overridden in-code or via env). |
| `SYNTH_GROUND_TRUTH_SEED` | Same chunk-sampling seed semantics as Step 6b. |
| `OPENAI_GT_MODEL` | Chat model id (default `gpt-4o-mini`). |
| `OPENAI_GT_BASE_URL` | Chat Completions base URL. |

> Compare OpenAI vs local drafts if you enable both—but promote rows to gold only after human review against the PDF.


In [18]:
# OPTIONAL — Step 6c: two-phase synthetic Q/A via OpenAI-compatible API (standalone prompts).
import random

USE_OPENAI_GT_LABELER = os.getenv("USE_OPENAI_GT_LABELER", "0").lower() in ("1", "true", "yes")
OPENAI_GT_MODEL = os.getenv("OPENAI_GT_MODEL", "gpt-4o-mini")
OPENAI_GT_BASE_URL = os.getenv("OPENAI_GT_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"))
OPENAI_GT_API_KEY = os.getenv("OPENAI_API_KEY", "")

N_SYNTH_GROUND_TRUTH = 20
N_SYNTH_GROUND_TRUTH = int(
    os.getenv(
        "N_SYNTH_GROUND_TRUTH",
        os.getenv("N_STEP6C_SYNTH_PAIRS", os.getenv("N_STEP6B_SYNTH_PAIRS", str(N_SYNTH_GROUND_TRUTH))),
    )
)
SYNTH_GROUND_TRUTH_SEED = int(
    os.getenv("SYNTH_GROUND_TRUTH_SEED", os.getenv("SYNTH6C_RANDOM_SEED", os.getenv("SYNTH6B_RANDOM_SEED", "42")))
)

SYNTH_Q_PROMPT = """
You write evaluation questions for a regulatory / technical document.
Given PASSAGE text only, write ONE clear, specific question that can be answered using ONLY information in the passage.
Rules:
- Output ONLY the question text (no numbering, no quotes, no preamble).
- Do not mention "the passage" or "the document".

PASSAGE:
{passage}
""".strip()


SYNTH_A_PROMPT = """
You write reference answers for RAG evaluation (ground-truth style labels).
Given PASSAGE and a QUESTION, write ONE concise factual paragraph answering the question using ONLY the passage.
Rules:
- If the passage does not support an answer, reply with exactly this single line: INSUFFICIENT_CONTEXT
- Max ~120 words, plain text, no bullet list.

QUESTION:
{question}

PASSAGE:
{passage}
""".strip()



def _trim_passage_c(passage: str, limit: int = 12000) -> str:
    t = passage.strip()
    if len(t) > limit:
        return t[:limit] + "\n\n[...truncated for labeling prompt...]"
    return t


if USE_OPENAI_GT_LABELER:
    if not OPENAI_GT_API_KEY:
        raise RuntimeError("USE_OPENAI_GT_LABELER is set but OPENAI_API_KEY is empty.")
    if "chunks" not in globals() or not chunks:
        raise RuntimeError("Run Step 3 so `chunks` exists before Step 6c.")

    llm_labeler = ChatOpenAI(
        model=OPENAI_GT_MODEL,
        api_key=OPENAI_GT_API_KEY,
        base_url=OPENAI_GT_BASE_URL,
        temperature=0,
    )

    rng = random.Random(SYNTH_GROUND_TRUTH_SEED)
    _chunk_list = list(chunks)
    n_pick = min(N_SYNTH_GROUND_TRUTH, len(_chunk_list))
    picked = rng.sample(_chunk_list, n_pick)
    passages_raw = [c.page_content for c in picked]
    page_hints = [c.metadata.get("page") for c in picked]
    passages = [_trim_passage_c(p) for p in passages_raw]

    synth_questions: list[str] = []
    for passage_trim, page_hint in tqdm(
        list(zip(passages, page_hints)),
        desc="Step 6c phase 1 — questions only (PDF chunks)",
    ):
        q_prompt = SYNTH_Q_PROMPT.format(passage=passage_trim)
        synth_questions.append(llm_labeler.invoke(q_prompt).content.strip())

    print("Phase 1 complete:", len(synth_questions), "questions (OpenAI); starting phase 2 — answers.")
    display(
        pd.DataFrame(
            {
                "question": synth_questions,
                "source_page": page_hints,
                "passage_chars": [len(p) for p in passages],
            }
        )
    )

    rows: list[dict] = []
    for question, passage_trim, page_hint in tqdm(
        list(zip(synth_questions, passages, page_hints)),
        desc="Step 6c phase 2 — answers from frozen questions + passages",
    ):
        a_prompt = SYNTH_A_PROMPT.format(question=question, passage=passage_trim)
        ground_truth = llm_labeler.invoke(a_prompt).content.strip()
        rows.append(
            {
                "question": question,
                "ground_truth": ground_truth,
                "source_page": page_hint,
                "passage_chars": len(passage_trim),
                "label_source": "step6c_synthetic_openai",
                "labeler_model": OPENAI_GT_MODEL,
            }
        )

    openai_gt_draft_df = pd.DataFrame(rows)
    print("Synthetic rows requested:", N_SYNTH_GROUND_TRUTH, "| produced:", len(openai_gt_draft_df))
    display(openai_gt_draft_df.head())
else:
    openai_gt_draft_df = pd.DataFrame(
        columns=[
            "question",
            "ground_truth",
            "source_page",
            "passage_chars",
            "label_source",
            "labeler_model",
        ]
    )
    print("Step 6c disabled (set USE_OPENAI_GT_LABELER=1 and OPENAI_API_KEY to run).")


Step 6c disabled (set USE_OPENAI_GT_LABELER=1 and OPENAI_API_KEY to run).


### Improving ground truth quality (important)

- **Step 6a (manual):** read the exact passages in the PDF, rewrite each `ground_truth` to stay tightly evidence-backed, keep internal page/section notes, and add a **second reviewer** when possible.
- **Step 6b (local synthetic):** after phase 1, skim generated **questions** for vagueness or off-topic wording; after phase 2, check each **answer** against the PDF chunk (and full context if needed). Watch for `INSUFFICIENT_CONTEXT` and hallucinations before treating any row as gold.
- **Step 6c (OpenAI synthetic, optional):** same review discipline as 6b, plus API cost and data-handling policies.

This workflow materially increases the reliability of Ragas scores and of any conclusions you draw about pipeline changes.


In [19]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

manual_gt_path = output_dir / "ground_truth_dataset_manual.csv"
ground_truth_df.to_csv(manual_gt_path, index=False)

llm_draft_gt_path = output_dir / "ground_truth_dataset_llm_draft.csv"
llm_draft_ground_truth_df.to_csv(llm_draft_gt_path, index=False)

if not openai_gt_draft_df.empty:
    openai_gt_path = output_dir / "ground_truth_dataset_openai_draft.csv"
    openai_gt_draft_df.to_csv(openai_gt_path, index=False)
    print("Saved:", openai_gt_path)

print("Saved:", manual_gt_path)
print("Saved:", llm_draft_gt_path)

Saved: outputs/ground_truth_dataset_manual.csv
Saved: outputs/ground_truth_dataset_llm_draft.csv


## Step 7 - Generate RAG answers for the evaluation set

Ground-truth labels are read from the **two CSV files** written in Step 6 (same folder as `outputs/ground_truth_dataset_manual.csv`):

| File | Role |
|------|------|
| `ground_truth_dataset_manual.csv` | Manual questions + reference answers (Step 6a) |
| `ground_truth_dataset_llm_draft.csv` | LLM-synthetic questions + reference answers (Step 6b) |

Optional environment overrides (paths must exist before this cell runs):

- `GROUND_TRUTH_MANUAL_CSV` — defaults to `outputs/ground_truth_dataset_manual.csv`
- `GROUND_TRUTH_LLM_CSV` — defaults to `outputs/ground_truth_dataset_llm_draft.csv`

We run the **same** RAG pipeline on every row in **each** file (two scoring tracks). Retriever output depends only on the **question**; the reference string for metrics comes from the CSV row.


In [20]:
# --- Step 7: load ground truth from CSV (manual + LLM), then run RAG for each track ---
import os


def _gt_rows_from_csv(path: Path, *, label: str) -> list[dict]:
    """Load rows with ``question`` and ``ground_truth`` only."""
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {label} CSV: {path}. Run the Step 6 export cell first, or set GROUND_TRUTH_* env override."
        )
    df = pd.read_csv(path)
    need = {"question", "ground_truth"}
    if not need.issubset(df.columns):
        raise ValueError(f"{path} must contain columns {sorted(need)}; got {list(df.columns)}")
    df = df[list(need)].dropna(subset=list(need))
    return df.to_dict("records")


def _rag_eval_rows(gt_rows: list[dict], retriever, tqdm_desc: str) -> list[dict]:
    out: list[dict] = []
    for row in tqdm(gt_rows, desc=tqdm_desc):
        rag_out = answer_with_rag(row["question"], retriever)
        out.append(
            {
                "question": row["question"],
                "answer": rag_out["answer"],
                "contexts": rag_out["contexts"],
                "ground_truth": row["ground_truth"],
            }
        )
    return out


_gt_root = Path(os.getenv("GROUND_TRUTH_OUTPUT_DIR", "outputs"))
manual_gt_path = Path(os.getenv("GROUND_TRUTH_MANUAL_CSV", _gt_root / "ground_truth_dataset_manual.csv"))
llm_gt_path = Path(os.getenv("GROUND_TRUTH_LLM_CSV", _gt_root / "ground_truth_dataset_llm_draft.csv"))

manual_gt_rows = _gt_rows_from_csv(manual_gt_path, label="manual")

if llm_gt_path.is_file():
    _llm_df = pd.read_csv(llm_gt_path)
    if {"question", "ground_truth"}.issubset(_llm_df.columns):
        _llm_df = _llm_df[["question", "ground_truth"]].dropna()
        llm_gt_rows = _llm_df.to_dict("records")
    else:
        llm_gt_rows = []
        print("Warning: LLM CSV missing question/ground_truth columns — LLM track skipped.")
else:
    llm_gt_rows = []
    print(f"Note: LLM CSV not found at {llm_gt_path} — LLM scoring track skipped (optional file).")

eval_df_manual = pd.DataFrame(
    _rag_eval_rows(manual_gt_rows, retriever, tqdm_desc="Step 7 RAG (manual GT CSV)")
)
eval_df_llm = (
    pd.DataFrame(_rag_eval_rows(llm_gt_rows, retriever, tqdm_desc="Step 7 RAG (LLM GT CSV)"))
    if llm_gt_rows
    else pd.DataFrame(columns=["question", "answer", "contexts", "ground_truth"])
)

eval_df = eval_df_manual
eval_rows = eval_df_manual.to_dict("records")

print("Manual GT rows:", len(eval_df_manual), "from", manual_gt_path)
print("LLM GT rows:", len(eval_df_llm), "from", llm_gt_path if llm_gt_path.is_file() else "(n/a)")

display(eval_df_manual[["question", "answer", "ground_truth"]].head())
if len(eval_df_llm):
    display(eval_df_llm[["question", "answer", "ground_truth"]].head())


Step 7 RAG (LLM GT CSV): 100%|██████████| 20/20 [00:43<00:00,  2.16s/it]

Manual GT rows: 6 from outputs/ground_truth_dataset_manual.csv
LLM GT rows: 20 from outputs/ground_truth_dataset_llm_draft.csv


,question,answer,ground_truth
0,What is the main objective of the IGF Code?,The main objective of the IGF Code is to address all areas that need special consideration for the usage of low-flashpoint fuels and provide a goa...,"The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework."
1,Who are the intended stakeholders or users of the IGF Code?,The intended stakeholders or users of the IGF Code are:\n\n* Contracting Governments to the Convention\n* Members of the Organization which are no...,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles."
2,What kinds of behavior or practices does the Code try to prevent?,I don't have enough information from the provided document to answer your question about what kinds of behavior or practices the Code tries to pre...,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance."
3,"Does the Code include governance or compliance expectations? If yes, what is the intent?","I don't have enough information from the provided document to determine if the Code includes governance or compliance expectations, or what the in...","Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations."
4,How does the Code describe accountability when rules are not followed?,I don't have enough information from the provided document to answer your question about how the Code describes accountability when rules are not ...,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes."


,question,answer,ground_truth
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure?,"According to the provided context, specifically page 102 of MSC 95/22/Add.1 Annex 1, the method for carrying out buckling strength analyses of fue...","The buckling strength analyses of fuel tanks subject to external pressure shall be carried out in accordance with recognized standards, which adeq..."
1,"What is the minimum distance between fuel tanks in a cargo ship with a volume Vc greater than or equal to 30,000 m3?","For a cargo ship with a volume Vc greater than or equal to 30,000 m3, the minimum distance between fuel tanks is 2 meters.","For a cargo ship with a volume Vc greater than or equal to 30,000 m3, the minimum distance between fuel tanks is 2 meters, as specified in paragra..."
2,What page number is referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces?,9.7,The page number referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces is 78.
3,What is the minimum design condition under which a type expansion joint must withstand twice the design pressure?,I don't have enough information from the provided document to answer your question. The context only mentions design conditions and requirements f...,The minimum design condition under which a type expansion joint must withstand twice the design pressure is at the minimum design temperature. Thi...
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.


In [21]:
raw_eval_path_manual = output_dir / "rag_eval_raw_outputs_manual_gt.csv"
temp_m = eval_df_manual.copy()
temp_m["contexts"] = temp_m["contexts"].apply(lambda x: "\n\n---\n\n".join(x))
temp_m.to_csv(raw_eval_path_manual, index=False)
print("Saved:", raw_eval_path_manual)

if len(eval_df_llm):
    raw_eval_path_llm = output_dir / "rag_eval_raw_outputs_llm_gt.csv"
    temp_l = eval_df_llm.copy()
    temp_l["contexts"] = temp_l["contexts"].apply(lambda x: "\n\n---\n\n".join(x))
    temp_l.to_csv(raw_eval_path_llm, index=False)
    print("Saved:", raw_eval_path_llm)


Saved: outputs/rag_eval_raw_outputs_manual_gt.csv
Saved: outputs/rag_eval_raw_outputs_llm_gt.csv


## Step 8 - Evaluate with Ragas metrics

We compute **Faithfulness** and **Answer relevancy** for **each** CSV ground-truth track (manual first, then LLM when that file has rows). Compare aggregates to see how scores shift when the reference label source changes.

### Metrics being used

- **Faithfulness** (`faithfulness`): measures whether the RAG answer is **supported by retrieved context**.
  - This metric asks: *"Does the answer add claims that are not grounded in the retrieved evidence?"*
  - A high score means most answer statements can be traced back to the contexts.
  - A low score usually indicates hallucination, over-inference, or weak retrieval coverage.

- **Answer Relevancy** (`answer_relevancy`): measures how well the answer **addresses the user’s question**.
  - This metric asks: *"Does the answer actually respond to what was asked?"*
  - A high score means the answer is focused on the question intent.
  - A low score often means the answer is off-topic, vague, or only partially addresses the question.

### How to interpret the scores

- Both metrics typically fall in the **0 to 1** range (higher is better).
- Do not rely on means alone; inspect low-score samples in Step 9 to find concrete failure modes.
- Quick reading by score pattern:
  - **High Faithfulness + High Relevancy**: healthy behavior (good grounding and good focus).
  - **High Faithfulness + Low Relevancy**: evidence is grounded, but generation/prompting misses the question intent.
  - **Low Faithfulness + High Relevancy**: answer sounds on-topic but lacks evidence support (hallucination risk).
  - **Low Faithfulness + Low Relevancy**: broader pipeline issues (retrieval, prompting, or both).

### Why compare manual GT vs LLM GT

- **Manual ground truth** is usually more trustworthy for final quality conclusions.
- **LLM ground truth** helps scale evaluation faster, but can inherit labeler-model bias.
- If the two tracks diverge a lot, prioritize manual inspection of high-delta samples before making decisions.


In [22]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

/tmp/ipykernel_20986/228170927.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_20986/228170927.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


In [23]:
# --- Ragas baseline (retriever k from Step 4): manual CSV vs LLM CSV ---
ragas_ready_manual = eval_df_manual[["question", "answer", "contexts", "ground_truth"]].copy()
ragas_dataset_manual = Dataset.from_pandas(ragas_ready_manual, preserve_index=False)
print("Manual GT — Dataset:", ragas_dataset_manual)

result_manual = evaluate(
    dataset=ragas_dataset_manual,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=embeddings,
)
score_df_manual = result_manual.to_pandas()

if len(eval_df_llm) == 0:
    result_llm = None
    score_df_llm = pd.DataFrame(columns=["faithfulness", "answer_relevancy"])
    print("Skipping LLM Ragas pass (no rows loaded from LLM CSV).")
else:
    ragas_ready_llm = eval_df_llm[["question", "answer", "contexts", "ground_truth"]].copy()
    ragas_dataset_llm = Dataset.from_pandas(ragas_ready_llm, preserve_index=False)
    print("LLM GT — Dataset:", ragas_dataset_llm)

    result_llm = evaluate(
        dataset=ragas_dataset_llm,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=embeddings,
    )
    score_df_llm = result_llm.to_pandas()

result = result_manual
score_df = score_df_manual


Manual GT — Dataset: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 6
})


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   8%|▊         | 1/12 [00:33<06:04, 33.10s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 12/12 [02:30<00:00, 12.57s/it]


LLM GT — Dataset: Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 20
})


Evaluating:   2%|▎         | 1/40 [00:19<12:26, 19.15s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   5%|▌         | 2/40 [00:50<16:42, 26.38s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  32%|███▎      | 13/40 [03:03<04:34, 10.17s/it]Exception raised in Job[10]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generati

In [24]:
display(score_df_manual.head())
if len(score_df_llm):
    display(score_df_llm.head())


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What is the main objective of the IGF Code?,"[Convention, \n \nHAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for \nShips using Gases or other Low-flas...",The main objective of the IGF Code is to address all areas that need special consideration for the usage of low-flashpoint fuels and provide a goa...,"The IGF Code establishes standards and guidance to ensure integrity, governance, and responsible conduct in the relevant IGF framework.",0.6,0.208876
1,Who are the intended stakeholders or users of the IGF Code?,"[Convention, \n \nHAVING CONSIDERED, at its ninety-fifth session, the draft International Code of Safety for \nShips using Gases or other Low-flas...",The intended stakeholders or users of the IGF Code are:\n\n* Contracting Governments to the Convention\n* Members of the Organization which are no...,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",0.5,1.000000
2,What kinds of behavior or practices does the Code try to prevent?,"[accordance with 18.4.3. \n \n.5 Where arrangements are provided for overriding the overflow control system, \nthey shall be such that inadvertent...",I don't have enough information from the provided document to answer your question about what kinds of behavior or practices the Code tries to pre...,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",0.2,0.000000
3,"Does the Code include governance or compliance expectations? If yes, what is the intent?","[5 REQUESTS the Secretary-General of the Organization to transmit certified copies of \nthe present resolution and the text of the IGF Code, conta...","I don't have enough information from the provided document to determine if the Code includes governance or compliance expectations, or what the in...","Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",0.0,0.000000
4,How does the Code describe accountability when rules are not followed?,[6.9 Regulations for the maintaining of fuel storage condition .............................. 62 \n6.10 Regulations on atmospheric control within ...,I don't have enough information from the provided document to answer your question about how the Code describes accountability when rules are not ...,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",0.5,0.000000


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What method shall be used to carry out buckling strength analyses of fuel tanks subject to external pressure?,"[MSC 95/22/Add.1 \nAnnex 1, page 51 \n \n \nI:\MSC\95\MSC 95-22-ADD.1 (E).docx \n6.4.15.3.3.2 Buckling criteria shall be as follows: \n \nThe thic...","According to the provided context, specifically page 102 of MSC 95/22/Add.1 Annex 1, the method for carrying out buckling strength analyses of fue...","The buckling strength analyses of fuel tanks subject to external pressure shall be carried out in accordance with recognized standards, which adeq...",NaN,0.984623
1,"What is the minimum distance between fuel tanks in a cargo ship with a volume Vc greater than or equal to 30,000 m3?",[tanks the distance shall be measured to the bulkheads surrounding the tank \ninsulation. \n \n.4 In no case shall the boundary of the fuel tank b...,"For a cargo ship with a volume Vc greater than or equal to 30,000 m3, the minimum distance between fuel tanks is 2 meters.","For a cargo ship with a volume Vc greater than or equal to 30,000 m3, the minimum distance between fuel tanks is 2 meters, as specified in paragra...",0.666667,0.883583
2,What page number is referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces?,[9.5 Regulations for fuel distribution outside of machinery space ........................ 77 \n9.6 Regulations for fuel supply to consumers in ga...,9.7,The page number referenced for regulations on gas fuel supply to consumers in ESD-protected machinery spaces is 78.,1.000000,0.185410
3,What is the minimum design condition under which a type expansion joint must withstand twice the design pressure?,"[MSC 95/22/Add.1 \nAnnex 1, page 35 \n \n \nI:\MSC\95\MSC 95-22-ADD.1 (E).docx \nwhere: \n \n x.st, y.st, z.st, xy.st, xz.st and yz.st are s...",I don't have enough information from the provided document to answer your question. The context only mentions design conditions and requirements f...,The minimum design condition under which a type expansion joint must withstand twice the design pressure is at the minimum design temperature. Thi...,NaN,0.000000
4,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,"[given in 6.4.15 for the various tank types, a vapour pressure 𝑃ℎ higher than \nP0 may be accepted for site specific conditions (harbour or other...",The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,0.500000,0.885009


In [26]:
summary = {
    "faithfulness_mean_manual_gt": float(score_df_manual["faithfulness"].mean()),
    "answer_relevancy_mean_manual_gt": float(score_df_manual["answer_relevancy"].mean()),
}
if len(score_df_llm):
    summary["faithfulness_mean_llm_gt"] = float(score_df_llm["faithfulness"].mean())
    summary["answer_relevancy_mean_llm_gt"] = float(score_df_llm["answer_relevancy"].mean())
else:
    summary["faithfulness_mean_llm_gt"] = None
    summary["answer_relevancy_mean_llm_gt"] = None

summary


{'faithfulness_mean_manual_gt': 0.3,
 'answer_relevancy_mean_manual_gt': 0.20147934902649492,
 'faithfulness_mean_llm_gt': 0.75,
 'answer_relevancy_mean_llm_gt': 0.6701126761749449}

In [ ]:
pd.DataFrame([summary]).T


## Step 9 - Inspect low-score samples (hallucination analysis)

Aggregate means are useful, but **row-level inspection** is where you identify concrete failure modes.

### What this step does

1. Builds per-row analysis tables by merging:
   - RAG outputs from Step 7 (`eval_df_manual`, `eval_df_llm`)
   - Ragas scores from Step 8 (`score_df_manual`, `score_df_llm`)
2. Sorts each table by low `faithfulness` and low `answer_relevancy` first.
3. Lets you inspect one row in detail (question, answer, ground truth, scores, and retrieved context).

### How to use this step effectively

- Start with **very low faithfulness** rows to find unsupported claims.
- Then review **low relevancy** rows to catch off-topic answers.
- Compare the **same pattern** across manual GT and LLM GT tracks.
- If both tracks flag the same question as weak, treat it as high-priority pipeline debt.

### Common diagnosis patterns

- **Low faithfulness + high relevancy**: answer sounds right but is not grounded in retrieved evidence.
- **High faithfulness + low relevancy**: answer is grounded, but does not address the question intent.
- **Low on both**: retrieval quality, prompt constraints, or both likely need improvement.

In [29]:
def _build_analysis_df(eval_df: pd.DataFrame, score_df: pd.DataFrame, track_name: str) -> pd.DataFrame:
    if eval_df.empty or score_df.empty:
        return pd.DataFrame()

    out = pd.concat(
        [
            eval_df.reset_index(drop=True),
            score_df[["faithfulness", "answer_relevancy"]].reset_index(drop=True),
        ],
        axis=1,
    )
    out["gt_track"] = track_name
    out = out.sort_values(by=["faithfulness", "answer_relevancy"], ascending=[True, True]).reset_index(drop=True)
    return out


analysis_df_manual = _build_analysis_df(eval_df_manual, score_df_manual, "manual_csv")
analysis_df_llm = _build_analysis_df(eval_df_llm, score_df_llm, "llm_csv")

print("Lowest-score samples — manual GT (from CSV):")
if analysis_df_manual.empty:
    print("No manual rows available.")
else:
    display(
        analysis_df_manual[
            ["question", "answer", "ground_truth", "faithfulness", "answer_relevancy", "gt_track"]
        ].head(5)
    )

print("Lowest-score samples — LLM GT (from CSV):")
if analysis_df_llm.empty:
    print("No LLM rows available (LLM CSV missing or empty).")
else:
    display(
        analysis_df_llm[
            ["question", "answer", "ground_truth", "faithfulness", "answer_relevancy", "gt_track"]
        ].head(5)
    )

# Combined view for quick filtering/comparison across tracks.
analysis_df_all = pd.concat([analysis_df_manual, analysis_df_llm], ignore_index=True)

# Backward-compatible alias used by older cells.
analysis_df = analysis_df_manual if not analysis_df_manual.empty else analysis_df_llm

Lowest-score samples — manual GT (from CSV):


,question,answer,ground_truth,faithfulness,answer_relevancy,gt_track
0,"Does the Code include governance or compliance expectations? If yes, what is the intent?","I don't have enough information from the provided document to determine if the Code includes governance or compliance expectations, or what the in...","Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.",0.0,0.0,manual_csv
1,What is the expected benefit of following the IGF Code consistently?,I don't have enough information from the provided document to answer your question about the expected benefit of following the IGF Code consistently.,"Consistent adherence improves trust, transparency, and responsible decision-making across the organizations and participants covered by the Code.",0.0,0.0,manual_csv
2,What kinds of behavior or practices does the Code try to prevent?,I don't have enough information from the provided document to answer your question about what kinds of behavior or practices the Code tries to pre...,"It seeks to prevent misconduct, unethical behavior, conflicts of interest, and practices that undermine integrity and compliance.",0.2,0.0,manual_csv
3,How does the Code describe accountability when rules are not followed?,I don't have enough information from the provided document to answer your question about how the Code describes accountability when rules are not ...,"The Code emphasizes accountability mechanisms so non-compliance can be identified, addressed, and corrected through defined processes.",0.5,0.0,manual_csv
4,Who are the intended stakeholders or users of the IGF Code?,The intended stakeholders or users of the IGF Code are:\n\n* Contracting Governments to the Convention\n* Members of the Organization which are no...,"The Code is intended for organizations, professionals, and stakeholders expected to align their behavior and decisions with the IGF principles.",0.5,1.0,manual_csv


Lowest-score samples — LLM GT (from CSV):


,question,answer,ground_truth,faithfulness,answer_relevancy,gt_track
0,What types of instrumentation and electrical apparatus should be installed in the specified areas?,"According to the provided context, instrumentation and electrical apparatus installed in the following areas should be suitable for zone 1:\n\n* T...","Instrumentation and electrical apparatus should be installed in the specified areas that are within 3 meters of fuel tank hatches, ullage openings...",0.166667,0.782281,llm_csv
1,What is the maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces?,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,The maximum design vapour pressure (P0) for Type B independent tanks primarily constructed of plane surfaces is less than 0.07 MPa.,0.500000,0.885009,llm_csv
2,What types of flanges are allowed for piping with design temperatures colder than minus 55°C?,"For piping with design temperatures colder than minus 55°C, only welded neck flanges are allowed.","For piping with design temperatures colder than minus 55°C, only welded neck flanges shall be used.",0.500000,0.997347,llm_csv
3,"What is considered ""fuel"" in the context of these regulations?","I don't have enough information from the provided document to determine what is considered ""fuel"" in the context of these regulations.","According to the regulations, ""fuel"" is considered to be natural gas, either in its liquefied or gaseous state. This definition applies throughout...",0.500000,NaN,llm_csv
4,What conditions must be met for a risk assessment to be required for ships to which part A-1 applies?,I don't have enough information from the provided document to answer your question about what conditions must be met for a risk assessment to be r...,"A risk assessment is required for ships to which part A-1 applies when explicitly required by specific regulations listed in the passage, includin...",0.666667,0.000000,llm_csv


In [31]:
track = "manual"  # choose: "manual", "llm", or "all"
row_id = 0

track_to_df = {
    "manual": analysis_df_manual,
    "llm": analysis_df_llm,
    "all": analysis_df_all,
}

if track not in track_to_df:
    raise ValueError(f"Invalid track={track!r}. Use one of {list(track_to_df)}")

picked_df = track_to_df[track]
if picked_df.empty:
    raise RuntimeError(f"No rows available for track={track!r}.")
if not (0 <= row_id < len(picked_df)):
    raise IndexError(f"row_id={row_id} out of range for track={track!r}; valid: 0..{len(picked_df)-1}")

row = picked_df.iloc[row_id]

print("Track:", row.get("gt_track", track))
print("Row:", row_id, "/", len(picked_df) - 1)
print("Question:\n", row["question"])
print("\nModel Answer:\n", row["answer"])
print("\nGround Truth:\n", row["ground_truth"])
print("\nFaithfulness:", row["faithfulness"])
print("Answer Relevancy:", row["answer_relevancy"])

print("\nFirst retrieved context snippet:\n")
print(row["contexts"][0][:1500] if row["contexts"] else "No context")

Track: manual_csv
Row: 0 / 5
Question:
 Does the Code include governance or compliance expectations? If yes, what is the intent?

Model Answer:
 I don't have enough information from the provided document to determine if the Code includes governance or compliance expectations, or what the intent is.

Ground Truth:
 Yes. It provides governance and compliance expectations to make responsibilities clear, improve accountability, and reduce policy violations.

Faithfulness: 0.0
Answer Relevancy: 0.0

First retrieved context snippet:

5 REQUESTS the Secretary-General of the Organization to transmit certified copies of 
the present resolution and the text of the IGF Code, contained in the annex, to all Contracting 
Governments to the Convention; 
 
6 REQUESTS ALSO the Secretary-General of the Org anization to transmit copies of 
the present resolution and the text of the IGF Code contained in the annex to all Members of 
the Organization which are not Contracting Governments to the SOLAS Conve

## Step 10 - Improve pipeline and re-evaluate

Typical improvements to reduce hallucinations:

- Increase retrieval depth (`k`)
- Improve chunking (`chunk_size`, `overlap`)
- Strengthen answer constraints in prompt
- Add stricter refusal behavior for missing evidence

In this exercise, we first test one simple change: increase `k` from 4 to 6.

In [32]:
retriever_k6 = vector_db.as_retriever(search_kwargs={"k": 6})
print("New retriever top-k:", retriever_k6.search_kwargs.get("k"))

New retriever top-k: 6


In [33]:
eval_df_k6_manual = pd.DataFrame(
    _rag_eval_rows(manual_gt_rows, retriever_k6, tqdm_desc="Step 10 RAG k=6 (manual GT CSV)")
)
eval_df_k6_llm = (
    pd.DataFrame(_rag_eval_rows(llm_gt_rows, retriever_k6, tqdm_desc="Step 10 RAG k=6 (LLM GT CSV)"))
    if llm_gt_rows
    else pd.DataFrame(columns=["question", "answer", "contexts", "ground_truth"])
)

eval_df_k6 = eval_df_k6_manual
ragas_dataset_k6_manual = Dataset.from_pandas(
    eval_df_k6_manual[["question", "answer", "contexts", "ground_truth"]].copy(),
    preserve_index=False,
)
result_k6_manual = evaluate(
    dataset=ragas_dataset_k6_manual,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=embeddings,
)
score_df_k6_manual = result_k6_manual.to_pandas()

if llm_gt_rows:
    ragas_dataset_k6_llm = Dataset.from_pandas(
        eval_df_k6_llm[["question", "answer", "contexts", "ground_truth"]].copy(),
        preserve_index=False,
    )
    result_k6_llm = evaluate(
        dataset=ragas_dataset_k6_llm,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=embeddings,
    )
    score_df_k6_llm = result_k6_llm.to_pandas()
else:
    score_df_k6_llm = pd.DataFrame(columns=["faithfulness", "answer_relevancy"])

ragas_dataset_k6 = ragas_dataset_k6_manual
result_k6 = result_k6_manual
score_df_k6 = score_df_k6_manual


Evaluating:  17%|█▋        | 2/12 [00:42<03:49, 22.94s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   2%|▎         | 1/40 [00:10<07:02, 10.82s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   5%|▌         | 2/40 [00:41<14:23, 22.72s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evalua

In [34]:
_has_llm = len(score_df_llm) > 0

{
    "faithfulness_mean_k4_manual_gt": float(score_df_manual["faithfulness"].mean()),
    "answer_relevancy_mean_k4_manual_gt": float(score_df_manual["answer_relevancy"].mean()),
    "faithfulness_mean_k4_llm_gt": float(score_df_llm["faithfulness"].mean()) if _has_llm else None,
    "answer_relevancy_mean_k4_llm_gt": float(score_df_llm["answer_relevancy"].mean()) if _has_llm else None,
    "faithfulness_mean_k6_manual_gt": float(score_df_k6_manual["faithfulness"].mean()),
    "answer_relevancy_mean_k6_manual_gt": float(score_df_k6_manual["answer_relevancy"].mean()),
    "faithfulness_mean_k6_llm_gt": float(score_df_k6_llm["faithfulness"].mean()) if _has_llm else None,
    "answer_relevancy_mean_k6_llm_gt": float(score_df_k6_llm["answer_relevancy"].mean()) if _has_llm else None,
}


{'faithfulness_mean_k4_manual_gt': 0.3,
 'answer_relevancy_mean_k4_manual_gt': 0.20147934902649492,
 'faithfulness_mean_k4_llm_gt': 0.75,
 'answer_relevancy_mean_k4_llm_gt': 0.6701126761749449,
 'faithfulness_mean_k6_manual_gt': 0.6666666666666666,
 'answer_relevancy_mean_k6_manual_gt': 0.6179561873214402,
 'faithfulness_mean_k6_llm_gt': 0.5240740740740741,
 'answer_relevancy_mean_k6_llm_gt': 0.757707384795917}

In [35]:
_has_llm = len(score_df_llm) > 0

comparison_df = pd.DataFrame(
    {
        "metric": ["faithfulness", "answer_relevancy"],
        "manual_gt_k4": [
            float(score_df_manual["faithfulness"].mean()),
            float(score_df_manual["answer_relevancy"].mean()),
        ],
        "manual_gt_k6": [
            float(score_df_k6_manual["faithfulness"].mean()),
            float(score_df_k6_manual["answer_relevancy"].mean()),
        ],
    }
)
comparison_df["delta_k6_vs_k4_manual"] = comparison_df["manual_gt_k6"] - comparison_df["manual_gt_k4"]

if _has_llm:
    comparison_df["llm_gt_k4"] = [
        float(score_df_llm["faithfulness"].mean()),
        float(score_df_llm["answer_relevancy"].mean()),
    ]
    comparison_df["llm_gt_k6"] = [
        float(score_df_k6_llm["faithfulness"].mean()),
        float(score_df_k6_llm["answer_relevancy"].mean()),
    ]
    comparison_df["delta_k6_vs_k4_llm"] = comparison_df["llm_gt_k6"] - comparison_df["llm_gt_k4"]
else:
    nan2 = [float("nan"), float("nan")]
    comparison_df["llm_gt_k4"] = nan2
    comparison_df["llm_gt_k6"] = nan2
    comparison_df["delta_k6_vs_k4_llm"] = nan2

comparison_df


,metric,manual_gt_k4,manual_gt_k6,delta_k6_vs_k4_manual,llm_gt_k4,llm_gt_k6,delta_k6_vs_k4_llm
0,faithfulness,0.300000,0.666667,0.366667,0.750000,0.524074,-0.225926
1,answer_relevancy,0.201479,0.617956,0.416477,0.670113,0.757707,0.087595


## Step 11 - Save evaluation artifacts

Files written here include per-track raw outputs from Step 7 and analysis CSVs keyed by **manual** vs **LLM** ground-truth source (both loaded from Step 6 exports).


In [36]:
analysis_path_manual = output_dir / "rag_eval_analysis_manual_gt.csv"
analysis_df_manual.to_csv(analysis_path_manual, index=False)

if len(analysis_df_llm):
    analysis_path_llm = output_dir / "rag_eval_analysis_llm_gt.csv"
    analysis_df_llm.to_csv(analysis_path_llm, index=False)
else:
    analysis_path_llm = None

comparison_path = output_dir / "rag_eval_metric_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("Saved:")
print("-", analysis_path_manual)
if analysis_path_llm:
    print("-", analysis_path_llm)
print("-", comparison_path)


Saved:
- outputs/rag_eval_analysis_manual_gt.csv
- outputs/rag_eval_analysis_llm_gt.csv
- outputs/rag_eval_metric_comparison.csv


## Step 12 - Practical checklist for learners

Use this checklist in class/lab:

1. Verify data loading and chunk quality.
2. Build baseline RAG and record scores.
3. Curate higher-quality ground truth samples.
4. Inspect low-faithfulness cases first.
5. Apply one pipeline change at a time.
6. Re-evaluate and compare deltas.
7. Keep an experiment log for reproducibility.

If you repeat this loop, learners will clearly see how evidence quality and retrieval quality impact hallucinations.

---

### Suggested extension exercises

- Compare multiple chunking strategies.
- Compare multiple prompt templates.
- Compare embedding models.
- Add more metrics (for example context precision/recall).
- Build a leaderboard table across experiments.